# Exercise 1.4 — Classify photos with a CNN (CIFAR-10) *(optional teaser)*

CIFAR-10: 60,000 **colour** images, 32×32 px, 10 classes (airplane, car, bird, cat, …).
Harder than MNIST: colour channels, varied backgrounds, real objects. Dense layers alone
struggle here — this is where **convolutional** layers earn their keep by exploiting
spatial structure with far fewer parameters.

This exercise is optional if time runs short. Training is much heavier than MNIST — in
Colab, switch on the GPU first: **Runtime → Change runtime type → GPU**.

Fill in the `TODO`s. Ready-made walkthrough, if you prefer to follow one:
[CNN_Cifar_10.ipynb (UNIFEI-IESTI01)](https://colab.research.google.com/github/Mjrovai/UNIFEI-IESTI01-TinyML-2022.1/blob/main/00_Curse_Folder/1_Fundamentals/Class_11/CNN_Cifar_10.ipynb)

> On Day 3 we build a 1-D CNN on raw vibration windows — same idea as this exercise, one
> dimension fewer, and small enough to run on your RAK4631.

In [ ]:
!pip install tensorflow matplotlib numpy

## Step 1 — Load and look

`keras.datasets.cifar10.load_data()` returns `(x_train, y_train), (x_test, y_test)`.
The class names, in label order:

```python
CLASSES = ["airplane", "automobile", "bird", "cat", "deer",
           "dog", "frog", "horse", "ship", "truck"]
```

Plot a 5×5 grid of training images with their class names, and note the shapes.

**Questions:**

- What shape does the input data have now — pixels *and* channels? How does that compare
  with MNIST?
- `y_train` has an awkward shape here. What is it, and what will you have to do about it?
- Look at the images at 32×32. Can *you* classify them all correctly?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras

CLASSES = ["airplane", "automobile", "bird", "cat", "deer",
           "dog", "frog", "horse", "ship", "truck"]

# TODO: load CIFAR-10 into (x_train, y_train), (x_test, y_test)

# TODO: print the shapes of all four arrays

# TODO: plot a 5x5 grid of images titled with their class name

## Step 2 — Normalise

Same move as MNIST: scale the pixels to floats in `[0, 1]`. Three channels this time,
but the code does not change.

In [ ]:
# TODO: scale x_train and x_test to float32 in [0, 1]

## Step 3 — Build the CNN

A small VGG-style stack. Each `Conv2D` learns a bank of little filters that slide over
the image; each `MaxPooling2D` halves the resolution, so later layers see more of the
picture at once.

| Layer | Output shape |
|---|---|
| `Conv2D(32, 3, activation="relu", input_shape=(32, 32, 3))` | 30×30×32 |
| `MaxPooling2D(2)` | 15×15×32 |
| `Conv2D(64, 3, activation="relu")` | 13×13×64 |
| `MaxPooling2D(2)` | 6×6×64 |
| `Conv2D(64, 3, activation="relu")` | 4×4×64 |
| `Flatten()` | 1024 |
| `Dense(64, activation="relu")` | 64 |
| `Dense(10, activation="softmax")` | 10 |

Compile with `adam`, `sparse_categorical_crossentropy` and `accuracy`.

**Before you train:** run `model.summary()` and answer — how many parameters does the
first `Conv2D` have? How many would a `Dense(32)` on the flattened 3072-pixel input have
had? That ratio is the entire argument for convolution.

In [ ]:
# TODO: build the CNN described above

# TODO: compile it

# TODO: print model.summary() and compare conv vs dense parameter counts

## Step 4 — Train

Train for ~10 epochs with `validation_data=(x_test, y_test)`, then plot training vs.
validation accuracy.

**Expected output:** ~65–70 % test accuracy after ~10 epochs. (State of the art is >99 %
— and needs millions of parameters. We will care about the other end of that curve.)

**Questions:**

- How many epochs until accuracy is "good"? What *is* a good accuracy, to you?
- The training and validation curves separate much earlier than on MNIST. Why is this
  dataset so much easier to overfit?

In [ ]:
# TODO: fit the model, keeping the History object

# TODO: plot training vs validation accuracy per epoch

## Step 5 — Where does it go wrong?

Evaluate on the test set, then build the confusion matrix and plot a few misclassified
images with their true and predicted labels.

**Questions:**

- Which classes get mixed up? Are the confusions sensible (cat/dog) or bizarre?
- Do the mistakes look like *your* mistakes when you looked at the 32×32 grid in Step 1?

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# TODO: evaluate on the test set

# TODO: predict, build and plot the confusion matrix (use CLASSES as tick labels)

# TODO: plot a few misclassified images with "true vs predicted" titles

## Step 6 — Would it fit on the device?

Count the parameters (`model.count_params()`) and work out the weight memory at 4 bytes
per float32, then again at 1 byte if the weights were quantised to int8 (Module 9).

Compare with the RAK4631's 1 MB flash / 256 KB RAM — and remember that the weights are
only part of it: the activations need RAM too, and the biggest intermediate tensor in
this model is 30×30×32 floats.

**Question:** what would you cut first to make it fit?

In [ ]:
# TODO: print the parameter count and the weight memory in KB at float32 and at int8

# TODO: compute the size of the largest activation tensor in bytes (float32)

## Observe & discuss

- What shape does the input data have now — pixels, channels?
- What is the purpose of the convolution and pooling layers?
- How many epochs until accuracy is "good"? What *is* a good accuracy, to you?
- What is an activation function, and how is ReLU implemented? (One line of C. Write it.)